# Multi-Agent Code Review with Gemma 4 on Cerebras

Build a code-review pipeline in which specialized reviewers analyze the same pull request in parallel, a second layer verifies their findings, and a "Judge" produces the final merge recommendation.

In this workshop, you will:

- Send a real pull-request diff to Gemma 4 31B using the Cerebras Inference API.
- Compare one general review with several specialized reviews.
- Run independent reviews concurrently to reduce end-to-end latency.
- Verify proposed findings to remove unsupported claims.
- Separate merge-blocking problems from advisory feedback.
- Measure latency, token usage, and model-call count.

This notebook makes live API calls, so results may vary slightly between runs. The replay notebook provides a deterministic presentation fallback.

In [ ]:
# Run this setup cell before Steps 1 and 2.
import os, pathlib, subprocess, sys

if not pathlib.Path("fixtures/pr_103.json").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/am-will/multi-agent-pr-review.git"], check=True)
    os.chdir("multi-agent-pr-review")

try:
    import cerebras.cloud.sdk
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "cerebras_cloud_sdk"]
    )

print("Live runtime ready. Add CEREBRAS_API_KEY in Colab Secrets before continuing.")

## 1 · Load the pull request and prepare the model input

We will review a real pull request containing a description, file metadata, and a complete code diff. This cell combines those inputs into the context sent to every reviewer.

The diff is capped at 175,000 characters so the prompt remains within the model's context window.

In [ ]:
import json, pathlib, os

# The setup cell changes into the cloned repository in Colab. Fall back to the
# canonical Colab path only when this cell is run independently.
cwd_root = pathlib.Path.cwd()
colab_root = pathlib.Path("/content/multi-agent-pr-review")
repo_root = cwd_root if (cwd_root / "fixtures/pr_103.json").exists() else colab_root
pr_json_path = repo_root / "fixtures/pr_103.json"
diff_path = repo_root / "fixtures/pr103.diff"

if not pr_json_path.exists() or not diff_path.exists():
    raise RuntimeError("Run the setup cell before Step 1 so the fixture files are available.")

PR = json.loads(pr_json_path.read_text())
FULL_DIFF = diff_path.read_text()
MAX_DIFF_CHARS = 175_000
DIFF = FULL_DIFF[:MAX_DIFF_CHARS]
PR_INPUT = f"PR #{PR['number']}: {PR['title']}\n\n{PR['description']}\n\nFULL DIFF:\n{DIFF}"

s = PR["stats"]
print(f"{PR['repo']} #{PR['number']} — {PR['title']}")
print(f"by @{PR['author']} · {s['files']} files · +{s['additions']} −{s['deletions']}\n")
print(PR["description"] + "\n")
for f in PR["files"][:8]:
    print(f"  {f['path']:<48} +{f['additions']:>5} −{f['deletions']}")
print(f"  … {len(PR['files']) - 8} more files")

if len(FULL_DIFF) > len(DIFF):
    print(f"Live context budget: reviewing {len(DIFF):,} of {len(FULL_DIFF):,} diff characters.")

## 2 · Establish a baseline with one reviewer

Before building the multi-agent pipeline, we run one performance-focused reviewer. This establishes the request format and reports latency, token usage, throughput, and findings from one model call.

The helper requests structured JSON and includes a quota guard. Cerebras can return HTTP 429 for either request-rate or token-rate exhaustion; this notebook reports the exact server reason and paces calls against conservative defaults of 5 requests/minute and 30,000 tokens/minute. Override those defaults with `CEREBRAS_RPM` and `CEREBRAS_TPM` Colab Secrets only when the Cerebras Limits page shows higher values for this project.

In [ ]:
import asyncio, json, math, os, time
from collections import deque
import cerebras.cloud.sdk as cerebras_sdk
from cerebras.cloud.sdk import AsyncCerebras

try:
    from google.colab import userdata
except ImportError:
    userdata = None

def secret(name):
    if userdata is not None:
        try:
            return userdata.get(name)
        except userdata.SecretNotFoundError:
            pass
    return os.environ.get(name)

API_KEY = secret("CEREBRAS_API_KEY")
MODEL_OVERRIDE = secret("CEREBRAS_MODEL")

if not API_KEY:
    raise RuntimeError("Add CEREBRAS_API_KEY to Colab Secrets, then rerun this cell.")

# The SDK also retries 429/5xx responses by default. Disable those retries so one
# logical call never turns into several hidden requests before our quota guard runs.
client = AsyncCerebras(api_key=API_KEY, max_retries=0, timeout=60.0)
catalog = await client.models.list()
AVAILABLE_MODELS = [model.id for model in catalog.data]

gemma_models = [
    model_id for model_id in AVAILABLE_MODELS
    if "gemma" in model_id.lower() and ("31" in model_id or "4" in model_id)
]
MODEL = MODEL_OVERRIDE or (
    "gemma-4-31b" if "gemma-4-31b" in AVAILABLE_MODELS
    else (gemma_models[0] if gemma_models else None)
)

if not MODEL:
    raise RuntimeError(
        "This key has no visible Gemma 4 31B endpoint. Add the provisioned endpoint "
        "ID as a CEREBRAS_MODEL Colab Secret. Available models: "
        + ", ".join(AVAILABLE_MODELS)
    )

if "gemma" not in MODEL.lower() and not MODEL_OVERRIDE:
    raise RuntimeError(f"Refusing to substitute a non-Gemma model: {MODEL}")

REQUESTS_PER_MINUTE = int(secret("CEREBRAS_RPM") or 5)
TOKENS_PER_MINUTE = int(secret("CEREBRAS_TPM") or 30_000)
MAX_COMPLETION_TOKENS = int(secret("CEREBRAS_MAX_OUTPUT") or 600)
RESPONSES, STAGE = [], {}

class QuotaLimiter:
    def __init__(self, rpm, tpm, window=60.0):
        self.rpm, self.tpm, self.window = rpm, tpm, window
        self.events = deque()
        self.lock = asyncio.Lock()

    def estimate(self, system, user, max_output):
        # Gemma measured about 3.1 characters/token on this notebook. Using 3.0
        # plus the full output allowance intentionally overestimates the request.
        return math.ceil((len(system) + len(user)) / 3.0) + max_output

    async def acquire(self, estimated_tokens):
        if estimated_tokens > self.tpm:
            raise RuntimeError(
                f"One request is estimated at {estimated_tokens:,} tokens, above the "
                f"configured {self.tpm:,} TPM limit. Reduce its context first."
            )
        announced = False
        while True:
            async with self.lock:
                now = time.monotonic()
                while self.events and now - self.events[0][0] >= self.window:
                    self.events.popleft()
                used = sum(tokens for _, tokens in self.events)
                if len(self.events) < self.rpm and used + estimated_tokens <= self.tpm:
                    self.events.append((now, estimated_tokens))
                    return
                wait = max(0.05, self.window - (now - self.events[0][0]) + 0.05)
                if not announced:
                    print(
                        f"Quota guard: waiting {wait:.1f}s "
                        f"({len(self.events)}/{self.rpm} requests, "
                        f"~{used:,}/{self.tpm:,} tokens in the rolling minute)"
                    )
                    announced = True
            await asyncio.sleep(wait)

limiter = QuotaLimiter(REQUESTS_PER_MINUTE, TOKENS_PER_MINUTE)

def json_contract(system):
    if "adjudicator" in system:
        return (
            'Return exactly one JSON object with this shape: '
            '{"verdict":"APPROVE or REQUEST_CHANGES","blocking":["FINDING-ID"],'
            '"summary":"concise final assessment"}.'
        )
    if "merge-gate" in system:
        return (
            'Return exactly one JSON object with this shape: '
            '{"finding_id":"the supplied finding ID","blocking":true,'
            '"reason":"concise gate rationale"}.'
        )
    if "critic" in system:
        return (
            'Return exactly one JSON object with this shape: '
            '{"finding_id":"the supplied finding ID",'
            '"verdict":"confirmed or rejected","reason":"concise audit rationale"}.'
        )
    return (
        'Return exactly one JSON object with this shape: '
        '{"findings":[{"id":"CATEGORY-1","severity":"high or medium or low",'
        '"title":"concise title","file":"path","line":1,'
        '"evidence":"verifiable evidence","reasoning":"why it matters"}]}. '
        'If there are no findings, return {"findings":[]}.'
    )

def error_details(exc):
    response = getattr(exc, "response", None)
    status = getattr(exc, "status_code", None)
    headers = dict(getattr(response, "headers", {}) or {})
    request_id = headers.get("x-request-id", "unknown")
    retry_after = float(headers.get("retry-after", 0) or 0)
    message, code = str(exc), None
    if response is not None:
        try:
            payload = response.json()
            message = payload.get("message") or payload.get("detail") or message
            code = payload.get("code")
        except Exception:
            pass
    return status, code, message, request_id, retry_after

async def review(system: str, user: str):
    contract = json_contract(system)
    full_system = system + "\n\n" + contract
    estimated_tokens = limiter.estimate(full_system, user, MAX_COMPLETION_TOKENS)

    for attempt in range(4):
        await limiter.acquire(estimated_tokens)
        t0 = time.perf_counter()
        try:
            resp = await client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": full_system},
                    {"role": "user", "content": user},
                ],
                response_format={"type": "json_object"},
                temperature=0.0,
                max_completion_tokens=MAX_COMPLETION_TOKENS,
            )
            RESPONSES.append(resp)
            return resp, time.perf_counter() - t0
        except cerebras_sdk.RateLimitError as exc:
            status, code, message, request_id, retry_after = error_details(exc)
            if attempt == 3:
                raise RuntimeError(
                    f"Cerebras 429 after quota-aware retries: {code or 'rate_limit'}: "
                    f"{message} (request {request_id})"
                ) from exc
            delay = max(60.1, retry_after)
            print(
                f"Cerebras 429: {code or 'rate_limit'}: {message}; "
                f"waiting {delay:.1f}s (request {request_id})"
            )
            await asyncio.sleep(delay)
        except cerebras_sdk.APIConnectionError as exc:
            if attempt == 3:
                raise
            delay = 2 ** attempt
            print(f"Cerebras connection error; retrying in {delay}s: {exc.__cause__}")
            await asyncio.sleep(delay)
        except cerebras_sdk.APIStatusError as exc:
            status, code, message, request_id, _ = error_details(exc)
            if status is None or status < 500 or attempt == 3:
                raise RuntimeError(
                    f"Cerebras API error {status}: {code or 'unknown'}: {message} "
                    f"(request {request_id})"
                ) from exc
            delay = 2 ** attempt
            print(f"Cerebras {status}; retrying in {delay}s (request {request_id})")
            await asyncio.sleep(delay)

print("Live model:", MODEL)
print(f"Quota guard: {REQUESTS_PER_MINUTE} RPM / {TOKENS_PER_MINUTE:,} TPM")

BASELINE_DIFF_CHARS = 30_000
BASELINE_INPUT = (
    f"PR #{PR['number']}: {PR['title']}\n\n{PR['description']}"
    f"\n\nREPRESENTATIVE DIFF EXCERPT:\n{DIFF[:BASELINE_DIFF_CHARS]}"
)
print(f"Baseline context: {min(len(DIFF), BASELINE_DIFF_CHARS):,} diff characters")

resp, dt = await review(
    "You are the performance reviewer on a PR review panel. Mandate: complexity regressions, "
    "N+1 patterns, memory growth, main-thread stalls. Return findings as JSON. "
    "Returning zero findings is acceptable.",
    BASELINE_INPUT,
)

u, t = resp.usage, resp.time_info
print(f"{resp.model} · {dt * 1000:.0f} ms round-trip · {u.prompt_tokens:,} tokens in / {u.completion_tokens} out")
print(f"decode: {u.completion_tokens / t.completion_time:,.0f} tok/s\n")
for f in json.loads(resp.choices[0].message.content)["findings"]:
    print(f"[{f['severity'].upper():<6}] {f['title']}")
    print(f"         {f['file']}:{f['line']}\n")

## 3 · Define specialized reviewer roles

A general reviewer must consider many concerns at once. Instead, this panel gives eight reviewers a focused responsibility. Specialization makes each prompt easier to follow and shows which reviewer produced each finding.

In [ ]:
AGENTS = {
    "security": (
        "Find authorization mistakes, injection risks, exposed secrets, and unsafe "
        "process or file-system interactions."
    ),
    "edge cases": (
        "Test assumptions using null values, empty inputs, repeated calls, malformed "
        "data, and hostile paths."
    ),
    "performance": (
        "Find unnecessary repeated work, N+1 operations, unbounded memory growth, "
        "and code that could block latency-sensitive execution."
    ),
    "maintainability": (
        "Report design choices that make the code materially harder to understand, "
        "debug, or modify. Ignore cosmetic style preferences."
    ),
    "change scope": (
        "Determine whether this is the smallest reasonable change and identify "
        "unrelated behavior that should be separated."
    ),
    "requirements alignment": (
        "Check whether the implementation matches the behavior described in the "
        "pull-request title and description."
    ),
    "tests": (
        "Check whether tests exercise the important new behavior, failure modes, "
        "and regressions introduced by the change."
    ),
    "compatibility and impact": (
        "Identify affected callers, changed contracts, migration requirements, "
        "and possible effects on build tooling or continuous integration."
    ),
}
for name, mandate in AGENTS.items():
    print(f"{name:<14} {mandate}")

## 4 · Run all specialized reviewers in quota-aware parallel waves

All eight reviewers receive the same compact pull-request context. Calls that fit inside the current Cerebras request and token budgets run concurrently; the quota guard automatically holds the remaining reviewers for the next rolling-minute window.

The shared excerpt samples both the beginning and end of the five largest source-file diffs, so reviewers see implementation and test code rather than an arbitrary slice from the start of the pull request.

In [ ]:
PANEL_FILE_COUNT = 5
PANEL_FILE_CHARS = 2_400

def file_diff(path):
    marker = f"diff --git a/{path} b/{path}"
    start = FULL_DIFF.find(marker)
    if start < 0:
        return ""
    next_start = FULL_DIFF.find("\ndiff --git ", start + len(marker))
    return FULL_DIFF[start:] if next_start < 0 else FULL_DIFF[start:next_start]

def compact_file_diff(path, budget=PANEL_FILE_CHARS):
    section = file_diff(path)
    if len(section) <= budget:
        return section
    half = budget // 2
    return section[:half] + "\n... middle of file diff omitted ...\n" + section[-half:]

source_files = [
    item for item in PR["files"]
    if pathlib.Path(item["path"]).suffix in {".rs", ".py", ".js", ".ts", ".tsx", ".sh"}
]
panel_paths = [item["path"] for item in source_files[:PANEL_FILE_COUNT]]
PANEL_DIFF = "\n\n".join(compact_file_diff(path) for path in panel_paths)
PANEL_INPUT = (
    f"PR #{PR['number']}: {PR['title']}\n\n{PR['description']}"
    f"\n\nSHARED SOURCE DIFF EXCERPT ({', '.join(panel_paths)}):\n{PANEL_DIFF}"
)

t0 = time.perf_counter()
panel = await asyncio.gather(*[
    review(
        f"You are the {name} reviewer on a PR review panel. Mandate: {mandate}. "
        "Return findings as JSON. Returning zero findings is acceptable.",
        PANEL_INPUT,
    )
    for name, mandate in AGENTS.items()
])
STAGE["panel"] = time.perf_counter() - t0

panel_resps = [resp for resp, _ in panel]
findings = []
for name, (resp, _) in zip(AGENTS, panel):
    batch = json.loads(resp.choices[0].message.content)["findings"]
    for index, finding in enumerate(batch, 1):
        finding["id"] = f"{name.upper().replace(' ', '_')}-{index}"
    findings.extend(batch)
    high = sum(f["severity"] == "high" for f in batch)
    print(f"{name:<14} {len(batch)} findings" + (f"  ({high} high)" if high else ""))

print(f"\n{len(findings)} raw findings · {len(panel)} calls · {STAGE['panel']:.2f} s wall")

## 5 · Verify every proposed finding

A second model call checks each proposed finding against a bounded excerpt of the relevant file. This avoids resending the entire 175,000-character diff for every finding, which a 30,000 TPM key can never accept.

Verification calls still use `asyncio.gather`; the shared quota guard releases only the calls that fit in each rolling-minute window.

In [ ]:
AUDIT_DIFF_CHARS = 9_000

def evidence_context(finding):
    path = str(finding.get("file", "")).strip()
    marker = f"diff --git a/{path} b/{path}"
    start = FULL_DIFF.find(marker) if path else -1
    if start < 0:
        start = 0
    return FULL_DIFF[start:start + AUDIT_DIFF_CHARS]

async def audit(f, vote=1, votes=1):
    tag = f" (pass {vote} of {votes})" if votes > 1 else ""
    resp, _ = await review(
        "You are the critic. Audit one finding from a first-pass reviewer. Verify the quoted "
        "evidence against the supplied relevant diff excerpt and reject unsupported claims.",
        f"Finding under audit: {f['id']}{tag}\n\n{json.dumps(f, indent=2)}"
        f"\n\nRELEVANT DIFF EXCERPT:\n{evidence_context(f)}",
    )
    return json.loads(resp.choices[0].message.content)

t0 = time.perf_counter()
verdicts = await asyncio.gather(*[audit(f) for f in findings])
STAGE["verification"] = time.perf_counter() - t0

confirmed_findings = [f for f, v in zip(findings, verdicts) if v["verdict"] == "confirmed"]
rejected_findings = {v["finding_id"]: v for v in verdicts if v["verdict"] == "rejected"}

print(f"{len(findings)} findings verified in quota-aware waves in {STAGE['verification']:.2f} s")
print(f"→ {len(confirmed_findings)} confirmed · {len(rejected_findings)} rejected\n")

if rejected_findings:
    for fid, result in list(rejected_findings.items())[:3]:
        print(f"REJECTED [{fid}]  {result['reason']}\n")
else:
    print("No findings were rejected in this live run.\n")

print("confirmed:", ", ".join(f["id"] for f in confirmed_findings) or "none")

## 6 · Decide which findings block the merge

A valid finding is not necessarily severe enough to stop a merge. This stage classifies each confirmed finding as either **blocking**, which must be addressed before merging, or **advisory**, which can be handled separately.

A final adjudicator combines those assessments into one merge recommendation and a concise summary.

In [ ]:
t0 = time.perf_counter()
gates = await asyncio.gather(*[
    review(
        "You are the merge-gate assessor. Decide whether this confirmed finding blocks the "
        "merge or ships as advisory. Return JSON.",
        f"Blocking assessment for finding {f['id']}\n\n{json.dumps(f, indent=2)}",
    )
    for f in confirmed_findings
])
gate = {g["finding_id"]: g for g in (json.loads(r.choices[0].message.content) for r, _ in gates)}

resp, _ = await review(
    "You are the adjudicator. You see only confirmed, deduplicated findings with gate "
    "assessments. Return a final verdict, the blocking list, and a summary as JSON.",
    json.dumps([{**f, "blocking": gate[f["id"]]["blocking"]} for f in confirmed_findings], indent=2),
)
final = json.loads(resp.choices[0].message.content)
STAGE["gate + adjudicate"] = time.perf_counter() - t0

print(f"VERDICT: {final['verdict']}\n")
for fid in final["blocking"]:
    f = next(f for f in confirmed_findings if f["id"] == fid)
    print(f"BLOCKING [{fid}] {f['title']}")
    print(f"  {gate[fid]['reason']}\n")
print(final["summary"])

## 7 · Measure latency, throughput, and model usage

This step summarizes the operational cost of the complete review: model calls, end-to-end time, tokens processed, and parallel panel time.

The serial comparison below is an estimate based on the configured reference endpoint speed. It is not a measured Cerebras benchmark.

In [ ]:
SERIAL_TPS, SERIAL_TTFT = 60, 0.6

pipeline_calls = len(RESPONSES) - 1  # excludes the Step 2 baseline
wall = sum(STAGE.values())
tokens = sum(r.usage.total_tokens for r in RESPONSES[1:])
serial_panel = sum(SERIAL_TTFT + r.usage.completion_tokens / SERIAL_TPS for r in panel_resps)

print(f"model calls per PR       {pipeline_calls}")
print(f"end to end               {wall:.2f} s")
print(f"tokens through Gemma      {tokens / 1e6:.2f} M")
print()
print("estimated time for the same 8 panel prompts on the reference endpoint")
print(f"({SERIAL_TPS} tok/s, {SERIAL_TTFT:.1f} s to first token): {serial_panel:.0f} s")
print(f"\nestimated speedup vs the serial panel: {serial_panel / STAGE['panel']:.1f}x")

## 8 · Optional: require agreement from multiple verifiers

One verifier can occasionally accept or reject a finding incorrectly. For higher-confidence reviews, run three independent verification calls for each finding and keep the majority decision.

This option triples the model-call count for verification. Because the calls run concurrently, the increase in wall-clock time may be smaller than the increase in total compute. The standard workshop run leaves it disabled. Set `RUN_MAJORITY_VOTE = True` to measure it.

In [ ]:
RUN_MAJORITY_VOTE = False

if RUN_MAJORITY_VOTE:
    CRITIC_VOTES = 3

    async def audit_majority(f):
        votes = await asyncio.gather(*[audit(f, vote=v + 1, votes=CRITIC_VOTES)
                                       for v in range(CRITIC_VOTES)])
        confirmed = sum(v["verdict"] == "confirmed" for v in votes) > CRITIC_VOTES // 2
        return votes, confirmed

    t0 = time.perf_counter()
    results3 = await asyncio.gather(*[audit_majority(f) for f in findings])
    dial_wall = time.perf_counter() - t0

    majority_confirmed = [f for f, (_, ok) in zip(findings, results3) if ok]
    changed_decisions = [
        (f, votes)
        for f, (votes, ok) in zip(findings, results3)
        if ok != (f in confirmed_findings)
    ]

    print(f"verification calls   {len(findings)} → {len(findings) * CRITIC_VOTES}     "
          f"wall: {STAGE['verification']:.2f} s → {dial_wall:.2f} s")
    print(f"confirmed findings   {len(confirmed_findings)} → {len(majority_confirmed)}\n")

    for f, votes in changed_decisions:
        print(f"DECISION CHANGED [{f['id']}] {f['title']}")
        for i, v in enumerate(votes, 1):
            print(f"   vote {i}: {v['verdict']:<9} — {v['reason']}")
        print()
else:
    print("Multi-verifier consensus skipped. Set RUN_MAJORITY_VOTE = True to run it.")